# QuantForge demo

Run a strategy through the C++ backtest engine from Python and plot the
equity curve. The heavy lifting (event loop, execution models, portfolio,
analytics) happens in compiled C++; here we just marshal bar data in and plot
results out.

**Prerequisites**: `pip install .[notebook]` from the repo root (installs the
compiled `quantforge` module plus pandas/matplotlib).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import quantforge as qf
print('quantforge', qf.__version__)

## 1. Load data

We use a reproducible synthetic daily series shipped in `examples/data`. To use
real data instead, run `python scripts/make_sample_data.py --source stooq
--symbol AAPL --out examples/data/aapl_daily.csv` and point `read_csv` at it.

In [ ]:
df = pd.read_csv('data/synth_daily.csv')
df['ts'] = pd.to_datetime(df['date']).astype('int64')  # nanoseconds since epoch
print(f'{len(df)} bars from {df.date.iloc[0]} to {df.date.iloc[-1]}')
df.head()

## 2. Run a backtest

A moving-average crossover with 1 bp fixed slippage and a per-share fee. The
`seed` makes the run exactly reproducible.

In [ ]:
res = qf.run_backtest(
    df,
    strategy='ma_crossover',
    strategy_params={'fast': 20, 'slow': 60},
    slippage={'kind': 'fixed_bps', 'bps': 1.0},
    fee={'kind': 'per_share', 'fee': 0.005},
    initial_cash=100_000.0,
    target_position=100.0,
    seed=7,
)

rep = res['report']
print(f"final equity : {res['final_equity']:,.2f}")
print(f"total return : {rep['total_return']*100:6.2f}%")
print(f"sharpe       : {rep['sharpe']:.3f}")
print(f"sortino      : {rep['sortino']:.3f}")
print(f"max drawdown : {rep['max_drawdown']*100:6.2f}%")
print(f"turnover     : {rep['turnover']:.2f}")
print(f"hit rate     : {rep['hit_rate']*100:6.2f}%")
print(f"fills        : {res['num_fills']}")

## 3. Plot the equity curve

In [ ]:
ts = pd.to_datetime(res['equity_ts'])
equity = res['equity']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True,
                               gridspec_kw={'height_ratios': [3, 1]})
ax1.plot(ts, equity, lw=1.3)
ax1.axhline(res['final_cash'] * 0 + equity[0], color='grey', ls='--', lw=0.8,
            label='start')
ax1.set_title('QuantForge equity curve - MA(20,60) crossover')
ax1.set_ylabel('equity')
ax1.legend(loc='upper left')

# Drawdown underwater plot.
running_peak = np.maximum.accumulate(equity)
drawdown = (equity - running_peak) / running_peak
ax2.fill_between(ts, drawdown * 100, 0, color='crimson', alpha=0.4)
ax2.set_ylabel('drawdown %')
ax2.set_xlabel('date')
plt.tight_layout()
plt.show()

## 4. A small parameter sweep

Each configuration is an independent, deterministic backtest. (In C++ these can
run across a thread pool with identical results regardless of thread count; here
we loop in Python for simplicity.)

In [ ]:
rows = []
for fast in (10, 20, 30):
    for slow in (50, 60, 90):
        if fast >= slow:
            continue
        r = qf.run_backtest(df, 'ma_crossover', {'fast': fast, 'slow': slow},
                            slippage={'kind': 'fixed_bps', 'bps': 1.0}, seed=7)
        rows.append({'fast': fast, 'slow': slow,
                     'sharpe': r['report']['sharpe'],
                     'return_%': r['report']['total_return'] * 100,
                     'max_dd_%': r['report']['max_drawdown'] * 100,
                     'fills': r['num_fills']})

grid = pd.DataFrame(rows).sort_values('sharpe', ascending=False).reset_index(drop=True)
grid

## 5. Reproducibility check

The same seed yields a bit-identical equity path.

In [ ]:
a = qf.run_backtest(df, 'mean_reversion', {'lookback': 20, 'entry_z': 1.5, 'exit_z': 0.5},
                    slippage={'kind': 'random_bps', 'min_bps': 1.0, 'max_bps': 5.0},
                    latency={'kind': 'random', 'min_ns': 0, 'max_ns': 10_000_000}, seed=123)
b = qf.run_backtest(df, 'mean_reversion', {'lookback': 20, 'entry_z': 1.5, 'exit_z': 0.5},
                    slippage={'kind': 'random_bps', 'min_bps': 1.0, 'max_bps': 5.0},
                    latency={'kind': 'random', 'min_ns': 0, 'max_ns': 10_000_000}, seed=123)
print('identical equity path:', bool(np.array_equal(a['equity'], b['equity'])))
print('identical final equity:', a['final_equity'] == b['final_equity'])